In [2]:
import dotenv
from langchain_core.messages import HumanMessage
import os
from langchain_openai import OpenAIEmbeddings

from ai.rag.embeddings import get_embedding_model

dotenv.load_dotenv("../../.env")

True

In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from ai.tools.tools import retrieve_info, save_info
from langchain.agents import create_agent
from ai.tools.model import get_model
from ai.rag.vectorstore import get_vector_store
from langchain_core.documents import Document

vector_store = get_vector_store()
vector_store.add_documents([
    Document(
        page_content="Any person in Duck T. company can retire at the age of 25 years",
        metadata={"source": "hr_policy.pdf"},
    )
])

agent = create_agent(
    model=get_model(),
    tools=[retrieve_info, save_info],
    system_prompt=(
        "You are a helpful assistant with a knowledge base about Duck T. company.\n"
        "- For questions about Duck T. company, use retrieve_info and answer only from what it returns. "
        "If it has no answer, say so.\n"
        "- When the user tells you new facts and wants them remembered, first check with retrieve_info "
        "whether it's already known, then use save_info. Save one fact per call.\n"
        "- Confirm briefly what you saved."
    ),
    checkpointer=InMemorySaver(),
)


def chat(text: str, thread_id: str = "1"):
    result = agent.invoke(
        {"messages": [HumanMessage(text)]},
        {"configurable": {"thread_id": thread_id}},
    )
    result["messages"][-1].pretty_print()

chat("Remember: Duck T. company gives every employee 30 days of paid vacation and a free duck.")
chat("How many vacation days do Duck T. employees get?", thread_id="2")

================================== Ai Message ==================================

Done. I saved: "Duck T. company offers every employee 30 days of paid vacation and a free duck." under the topic employee_benefits.
================================== Ai Message ==================================

Duck T. employees receive 30 days of paid vacation per year (plus a free duck).
